<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


# AG2 101 (AutoGen): Complete Tutorial

Estimated time needed: **45** minutes

This notebook provides a comprehensive guide to AG2 (formerly AutoGen) basic concepts with runnable examples and detailed explanations. AG2 is an open-source programming framework for building AI agents and facilitating cooperation among multiple agents to solve tasks.

## Table of Contents

<ol>
    <li><a href="#Setup-and-Import">Setup and Import</a></li>
    <li><a href="#Configure-API-Keys">Configure API Keys</a></li>
    <li>
        <a href="#Introduction-to-Agent-Concepts">Introduction to Agent Concepts</a>
        <ol>
            <li><a href="#Conversable-Agent">Conversable Agent</a></li>
            <li><a href="#Creating-Specialized-Agents">Creating Specialized Agents</a></li>
            <li><a href="#Built-in-Agent-Types">Built-in Agent Types</a></li>
        </ol>
    </li>
    <li><a href="#Human-in-the-Loop">Human-in-the-Loop</a></li>
    <li>
        <a href="#Agent-Orchestration-&-Multi-Agent-Systems-in-AG2">Agent Orchestration & Multi-Agent Systems in AG2</a>
        <ol>
            <li><a href="#GroupChat-and-GroupChatManager">GroupChat and GroupChatManager</a></li>
        </ol>
    </li>
    <li><a href="#Tools-and-Extensions">Tools and Extensions</a></li>
    <li><a href="#Structured-Outputs">Structured Outputs</a></li>
    <li><a href="#Best-Practices">Best Practices</a></li>
    <li><a href="#Conclusion">Conclusion</a></li>
    <li><a href="#Authors">Authors</a></li>
    
</ol>


## Setup and Import

Dependencies for this notebook are managed by the project-level `requirements.in` file. The notebook does not install packages at runtime; make sure your environment is created from the repository requirements before running it.


In [1]:
# Dependencies are managed in ../../requirements.in:
# ag2[openai], python-dotenv, pydantic, matplotlib, numpy
print("Using project requirements; no notebook-level package installation is needed.")


Using project requirements; no notebook-level package installation is needed.


In [2]:
# Import necessary modules
import json
import logging
import os
import random

from dotenv import load_dotenv
from pydantic import BaseModel

from autogen import AssistantAgent, ConversableAgent, LLMConfig, UserProxyAgent
from autogen.agentchat import run_group_chat
from autogen.agentchat.group import AgentTarget, TerminateTarget
from autogen.agentchat.group.patterns import RoundRobinPattern
from autogen.coding import LocalCommandLineCodeExecutor

# Load environment variables from .env
load_dotenv()

MODEL_NAME = os.getenv("OPENAI_MODEL", "gpt-5-nano")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    raise RuntimeError("Set OPENAI_API_KEY in your environment or .env file before running this notebook.")

llm_config = LLMConfig(
    {
        "api_type": "openai",
        "model": MODEL_NAME,
        "api_key": OPENAI_API_KEY,
    }
)

logging.getLogger("autogen.oai.client").setLevel(logging.ERROR)
print(f"AG2 configured for OpenAI model: {MODEL_NAME}")


AG2 configured for OpenAI model: gpt-5-nano


In [3]:
import logging

# Suppress API key format warning
logging.getLogger("autogen.oai.client").setLevel(logging.ERROR)

### Configure API Keys

Autogen supports various LLM providers. Let's set up configuration for OpenAI.  

**Note:** In this environment, you do not need to run this cell and you can skip to the next section. 


In [4]:
# The notebook uses OpenAI credentials loaded by python-dotenv.
# Create a .env file in the repository root or export these variables in your shell:
# OPENAI_API_KEY=your_api_key
# OPENAI_MODEL=gpt-5-nano

print("OpenAI credentials are loaded from the environment via python-dotenv.")


OpenAI credentials are loaded from the environment via python-dotenv.


## Introduction to Agent Concepts

We have several agent concepts in AG2 to help you build your AI agents. We introduce the most common ones here:

- **Conversable agent**: Agents that are able to send messages, receive messages and generate replies using GenAI models, non-GenAI tools, or human inputs.

- **Human in the loop**: Add human input to the conversation.

- **Orchestrating multiple agents**: Users can orchestrate multiple agents with built-in conversation patterns such as swarms, group chats, nested chats, sequential chats or customize the orchestration by registering custom reply methods.

- **Tools**: Programs that can be registered, invoked and executed by agents.

- **Advanced concepts**: AG2 supports more concepts such as structured outputs, RAG, code execution, and so on.


### Conversable Agent

The ConversableAgent is the fundamental building block of AG2, designed to enable seamless communication between AI entities. This core agent type handles message exchange and response generation, serving as the base class for all agents in the framework.

#### Key characteristics:
- **Communication**: Can send and receive messages
- **Processing**: Handles information and generates responses
- **Personality**: Defined by system messages
- **Flexibility**: Base for all other agent types

In the example below, we'll create a simple information validation workflow with two specialized agents that communicate with each other.

**Note**: `initiate_chat()` starts a conversation with another agent.


In [5]:
# Student agent
student = ConversableAgent(
    name="student",
    system_message="You are a curious student. Ask clear, specific questions to learn new concepts.",
    human_input_mode="NEVER",
    llm_config=llm_config,
)

# Tutor agent
tutor = ConversableAgent(
    name="tutor",
    system_message="You are a helpful tutor who gives concise beginner-friendly explanations.",
    human_input_mode="NEVER",
    llm_config=llm_config,
)

chat_result = student.initiate_chat(
    tutor,
    message="What is AG2 in one short paragraph?",
    max_turns=2,
    summary_method="reflection_with_llm",
)

print(chat_result.summary)


student (to tutor):

What is AG2 in one short paragraph?

--------------------------------------------------------------------------------
tutor (to student):

AG2 means different things in different contexts. In chemistry, AG2 (with Ag as the symbol for silver) can denote the diatomic molecule Ag2, i.e., two silver atoms bonded together, which appears in gas-phase studies and spectroscopy. In other fields it might be a model name, product code, software version, or course/code designation. If you tell me the field you have in mind, I’ll tailor a precise one‑paragraph definition.

--------------------------------------------------------------------------------
student (to tutor):

Which field would you like me to define AG2 in? For example, chemistry (the silver diatomic molecule) or software/versioning (a version or code name), education (a course/code), etc. If you’re unsure, I can give two quick one‑paragraph definitions—one for chemistry and one for software—to compare.

----------

#### Why Add summary_method=`"reflection_with_llm"`?

By enabling summary_method=`"reflection_with_llm"`, we allow the system to generate a final response that is not simply a direct answer, but a **reflection of a structured conversation** between multiple agents with defined roles.

In this case, instead of prompting an LLM directly with *“What is a neural network?”*, the system simulates a conversation where:
- A **student** asks a clear question,
- A **tutor** responds with a beginner-friendly explanation,
- And the LLM reflects on this interaction to generate a detailed and coherent summary.
  
This approach can also be extended to **solve complex tasks** by simulating a structured conversation between **expert agents** (for example, a data scientist, a software engineer, and a business strategist), each contributing from their expertise to build a better solution.

This often produces **higher-quality, context-rich answers**, as the reasoning is distributed and clarified through role-based collaboration.  

**Note:** You can try running the conversation without adding the line `summary_method="reflection_with_llm"` and see the difference. 


## Creating Specialized Agents

The power of ConversableAgent comes from customization. Let's create agents with different personalities and purposes:


In [6]:
# Create specialized agents that share the same dotenv-backed OpenAI config.
tech_expert = ConversableAgent(
    name="tech_expert",
    system_message=(
        "You are a senior software engineer with expertise in Python, AI, and system design. "
        "Give technical, practical explanations and mention tradeoffs."
    ),
    llm_config=llm_config,
    human_input_mode="NEVER",
)

creative_writer = ConversableAgent(
    name="creative_writer",
    system_message=(
        "You are a creative writer. Explain technical topics using vivid but concise analogies."
    ),
    llm_config=llm_config,
    human_input_mode="NEVER",
)

user = ConversableAgent(name="user", llm_config=False, human_input_mode="NEVER")

for agent in [tech_expert, creative_writer]:
    result = user.initiate_chat(
        agent,
        message="Explain why multi-agent systems need clear roles.",
        max_turns=2,
        summary_method="reflection_with_llm",
    )
    print()
    print(f"--- {agent.name} summary ---")
    print(result.summary)


user (to tech_expert):

Explain why multi-agent systems need clear roles.

--------------------------------------------------------------------------------
tech_expert (to user):

Short answer: clear roles reduce coordination complexity, improve predictability, and enable scalable, reliable behavior in multi-agent systems. Without well-defined roles, agents interfere, duplicate work, or become idle, and verification becomes near impossible.

What “roles” buy you in MAS design

- Specialization and focus
  - Each role embodies a distinct responsibility (e.g., planner, executor, monitor). Specialization lets agents optimize for their task (latency, accuracy, energy) and reuse implementations across tasks.

- Clear interfaces and contracts
  - Roles come with defined inputs, outputs, and expected invariants. This reduces ambiguity, makes components composable, and simplifies testing and verification.

- Decentralized coordination with clear authority
  - Roles establish who makes certain 

## Built-in Agent Types

AG2 provides specialized agent classes built on `ConversableAgent` to streamline common workflows such as task-solving, tool use, and user interaction.

#### AssistantAgent — Task-solving LLM assistant

`AssistantAgent` is a subclass of `ConversableAgent` configured with a default system message tailored for solving tasks using LLMs. It can suggest Python code blocks, offer debugging suggestions, and provide structured responses.

- `human_input_mode`: Defaults to `"NEVER"` — the assistant operates autonomously.
- `code_execution_config`: Defaults to `False` — it does **not execute code** itself.
- Designed to work collaboratively with other agents (for example, `UserProxyAgent`) that handle execution.

This agent excels at reasoning, planning, and generating code — and expects others to handle the execution layer.

#### UserProxyAgent — Executing code on behalf of the user

`UserProxyAgent` is a subclass of `ConversableAgent` that acts as a proxy for the human user. It is designed to **execute code**, simulate user decisions, and provide execution-based feedback to other agents like `AssistantAgent`.

- `human_input_mode`: Defaults to `"ALWAYS"` — prompts the user at every turn.
- `llm_config`: Defaults to `False` — no LLM responses unless explicitly configured.
- **Code execution is enabled by default.**

You can customize its behavior by:
- Registering an auto-reply function via `.register_reply()`.
- Overriding `.get_human_input()` to change how user input is gathered.
- Overriding `.execute_code_blocks()`, `.run_code()`, or `.execute_function()` to control code execution behavior.

These two agents are often paired: the `AssistantAgent` writes code, and the `UserProxyAgent` executes it.


In [7]:
# AssistantAgent + UserProxyAgent with local code execution explicitly disabled by default.
assistant = AssistantAgent(
    name="assistant",
    system_message="You are a helpful assistant who writes and explains Python code clearly.",
    llm_config=llm_config,
)

user_proxy = UserProxyAgent(
    name="user_proxy",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=5,
    code_execution_config=False,
)

result = user_proxy.initiate_chat(
    assistant,
    message="Write a short Python function that returns the square of a number. Do not execute code.",
    max_turns=2,
    summary_method="reflection_with_llm",
)

print(result.summary)


user_proxy (to assistant):

Write a short Python function that returns the square of a number. Do not execute code.

--------------------------------------------------------------------------------
assistant (to user_proxy):

def square(n):
    """Return the square of a number."""
    return n * n

--------------------------------------------------------------------------------
user_proxy (to assistant):



--------------------------------------------------------------------------------
assistant (to user_proxy):

def square(n):
    """Return the square of a number."""
    return n * n

--------------------------------------------------------------------------------

>>>>>>>> TERMINATING RUN (fc521cfc-8849-4e8c-861f-664668aeaa78): Maximum turns (2) reached
- User asked for a short Python function that returns the square of a number and to avoid executing code.
- Assistant provided a concise function: def square(n): """Return the square of a number.""" return n * n
- The user left a bla

## Human-in-the-Loop

AG2 makes integrating human feedback seamless through its human-in-the-loop functionality, allowing AI agents to collaborate with humans during workflows.  

This is crucial for:   

- **Critical decisions** requiring human judgment
- **High-stakes scenarios** with significant consequences
- **Regulatory compliance** requiring human oversight
- **Quality assurance** and validation

You can configure how and when human input is solicited using the `human_input_mode` parameter:

- ALWAYS: Requires human input for every response

- NEVER: Operates autonomously without human involvement

- TERMINATE: Only requests human input to end conversations

For convenience, AG2 provides the specialized `UserProxyAgent` class that automatically sets `human_input_mode` to ALWAYS and supports **code execution**. 

**important note**: Always use code execution functionality with caution and at your own discretion.


### Human-in-the-Loop Example: Bug Triage Bot

This example demonstrates how to use AG2’s `ConversableAgent` in `human_input_mode="ALWAYS"` to enable **human-in-the-loop workflows**.

We simulate a **bug triage assistant** (`triage_bot`) that classifies bug reports as either:
- Escalate (for example, critical crash or security issue),
- Close (for example, minor cosmetic issue),
- Medium priority (default for others).

For each classification, the assistant **asks the human agent for confirmation or correction**. This ensures the AI doesn’t act on high-impact decisions without oversight.

At the end, the assistant summarizes the triage results.

---

### Try these inputs when prompted

When you’re prompted to reply as the human agent, try responding with the following:

- **Confirm assistant’s suggestion**  
  `"Yes, escalate it."`  
  `"Closing this makes sense."`

- **Override assistant’s suggestion**  
  `"This should be marked as high priority instead."`  
  `"Let’s keep this open for now."`

- **Ask for clarification**  
  `"Why do you think this is low priority?"`  
  `"Can you provide more reasoning?"`

You can also type `exit` at any time to end the conversation.


In [9]:
# Simulated human-in-the-loop triage: the agent asks for review, but the notebook stays non-blocking.
triage_system_message = """
You are a bug triage assistant. Classify each bug as low, medium, or high priority.
Ask for human confirmation in your response, then stop.
"""

triage_agent = ConversableAgent(
    name="bug_triage_agent",
    system_message=triage_system_message,
    llm_config=llm_config,
    human_input_mode="NEVER",
)

reviewer = ConversableAgent(name="reviewer", llm_config=False, human_input_mode="NEVER")

bug_report = random.choice([
    "The app crashes when users upload a CSV file larger than 20 MB.",
    "The settings page has a typo in the notification label.",
    "Users sometimes receive duplicate email receipts after checkout.",
])

result = reviewer.initiate_chat(
    triage_agent,
    message=f"Triage this bug report: {bug_report}",
    max_turns=2,
    summary_method="reflection_with_llm",
)

print(result.summary)


reviewer (to bug_triage_agent):

Triage this bug report: Users sometimes receive duplicate email receipts after checkout.

--------------------------------------------------------------------------------
bug_triage_agent (to reviewer):

- Priority: High

- Rationale: This is a user-facing issue that directly affects communication after checkout. Duplicate receipts can confuse customers, increase support load, and may raise trust or compliance concerns. It’s intermittent, which suggests a problematic edge case in the email sending or deduplication logic, making it important to fix promptly.

Please confirm whether you want this classified as High priority, or adjust if you have different guidelines.

--------------------------------------------------------------------------------
reviewer (to bug_triage_agent):



--------------------------------------------------------------------------------
bug_triage_agent (to reviewer):

Proposed priority: High

Rationale:
- This is a user-facing i

## Agent Orchestration & Multi-Agent Systems in AG2<a name="agent-orchestration"></a>

AG2 enables the coordination of multiple intelligent agents to collaboratively solve complex tasks. This is known as **agent orchestration** — a powerful design pattern where each agent plays a specialized role, and a **group manager** handles the conversation flow.

### Why Multi-Agent Systems?

Many real-world problems require more than just a single AI assistant. With AG2, you can:

- Assign specific roles and responsibilities to different agents.
- Orchestrate conversations using built-in patterns (for example, Auto, RoundRobin, Manual).
- Enable agents to build on each other's outputs and refine ideas.
- Incorporate human agents for oversight, decisions, or approval.

### Orchestration Patterns in AG2

AG2 provides several orchestration patterns to structure agent interactions:

- **Two-Agent Chat**: Simple back-and-forth between two agents.
- **Sequential Chat**: Conversations where one agent’s output becomes another’s input.
- **Group Chat**: Multiple agents interact, with selection logic for who speaks next.
- **Nested Chat**: Reusable sub-conversations packaged as a single workflow.

---

## GroupChat and GroupChatManager

In AG2, multi-agent collaboration is coordinated using `GroupChat` and `GroupChatManager`.

### GroupChat

`GroupChat` defines a team of agents and how they interact in a shared conversation. It includes:

- **Agents**: A list of AI (or human) agents that participate in the group dialogue.
- **Speaker Selection Method**: Determines which agent speaks next. Options include:
  - `"auto"`: Uses the LLM to select the most contextually appropriate agent.
  - `"round_robin"`: Agents take turns in a fixed sequence.
  - `"manual"`: Human selects the next speaker.
  - `"random"`: Agents are chosen randomly.

This structure allows you to define collaborative workflows where agents build on each other’s contributions.

### GroupChatManager

`GroupChatManager` is responsible for managing the flow of the group conversation and it:

- Orchestrates message passing between agents.
- Decides when to stop the conversation (for example, based on a termination condition or turn limit).
- Leverages the speaker selection method defined in `GroupChat`.

It acts like a facilitator that ensures the conversation runs according to the logic you've configured.

### How It Works

1. A conversation is initiated by one of the agents (often a user or teacher agent).
2. The `GroupChatManager` uses the selected pattern (for example, AutoPattern) to determine which agent speaks next.
3. Agents take turns responding based on their roles and system messages.
4. The conversation ends either when a stop condition (for example, a message like "DONE") is met or when a maximum number of turns is reached.

This setup enables modular, role-based collaboration — ideal for use cases like lesson planning, research workflows, or multi-perspective decision-making.

To explore more orchestration patterns, visit the AG2 documentation at:  
https://docs.ag2.ai/latest/docs/user-guide/advanced-concepts/orchestration/group-chat/introduction/

---

### Example: Group Chat for Lesson Planning

This example shows how a teacher agent collaborates with a planner and a reviewer to create a lesson plan, using AG2’s `GroupChat` and `AutoPattern` to manage the conversation.

**Note:** The `is_termination_msg` parameter used for `teacher` agent defines a custom rule to end the conversation —  
in this case, the workflow stops when the teacher replies with "DONE!".


In [11]:
# Pattern-based group chat using the current AG2 orchestration API.
lesson_planner = ConversableAgent(
    name="planner_agent",
    system_message="Create a short lesson plan for 4th graders.",
    description="Makes lesson plans.",
    llm_config=llm_config,
    human_input_mode="NEVER",
)

lesson_reviewer = ConversableAgent(
    name="reviewer_agent",
    system_message="Review the plan and suggest up to 3 brief edits.",
    description="Reviews lesson plans.",
    llm_config=llm_config,
    human_input_mode="NEVER",
)

teacher = ConversableAgent(
    name="teacher_agent",
    system_message="Finalize the lesson plan and end the workflow.",
    description="Finalizes lesson plans.",
    llm_config=llm_config,
    human_input_mode="NEVER",
)

lesson_planner.handoffs.set_after_work(AgentTarget(lesson_reviewer))
lesson_reviewer.handoffs.set_after_work(AgentTarget(teacher))
teacher.handoffs.set_after_work(TerminateTarget())

result = run_group_chat(
    pattern=RoundRobinPattern(
        initial_agent=lesson_planner,
        agents=[lesson_planner, lesson_reviewer, teacher],
        group_manager_args={"llm_config": llm_config},
    ),
    messages="Create a 15-minute lesson plan about plant life cycles.",
    max_rounds=6,
)

result.process()
print(result.summary)


_User (to chat_manager):

Create a 15-minute lesson plan about plant life cycles.

--------------------------------------------------------------------------------

Next speaker: planner_agent

planner_agent (to chat_manager):

Here’s a ready-to-teach 15-minute lesson plan on plant life cycles for 4th graders.

Title: Plant Life Cycles
Grade: 4
Duration: 15 minutes

Objectives
- Students will name the stages of a typical flowering plant life cycle: Seed, Germination, Seedling, Adult/Mature Plant, Flowering/Seed Production.
- Students will place a sequence of life-cycle cards in the correct order.
- Students will describe one thing a plant needs at each stage.

Materials
- Life cycle poster or chart showing: Seed → Germination → Seedling → Mature Plant → Flowering/Seeds
- 8–12 laminated life-cycle cards (one card per stage with simple pictures and labels)
  - Seed, Germination, Seedling, Mature Plant, Flower/Seed
- A small bag or cup with a bean or lentil seed, damp cotton or paper towe

## Tools and Extensions<a name="tools"></a>

Tools extend agent capabilities beyond text conversations, allowing them to:
- Execute code
- Access external APIs
- Perform calculations
- Interact with databases
- Generate visualizations

Agents gain significant utility through tools as they provide access to external data, APIs, and functionality.  

Let's explore how to integrate tools with AG2 agents:


In [12]:
from typing import Annotated

# Tool functions are registered on the caller for LLM selection and on the executor for execution.
math_asker = AssistantAgent(
    name="math_asker",
    system_message="Use the registered tool to check whether numbers are prime. Explain the result briefly.",
    llm_config=llm_config,
)

math_checker = UserProxyAgent(
    name="math_checker",
    human_input_mode="NEVER",
    code_execution_config=False,
    max_consecutive_auto_reply=3,
)

@math_checker.register_for_execution()
@math_asker.register_for_llm(description="Check whether a positive integer is prime.")
def is_prime(n: Annotated[int, "Positive integer to test"]) -> str:
    if n < 2:
        return "No"
    for i in range(2, int(n**0.5) + 1):
        if n % i == 0:
            return "No"
    return "Yes"

result = math_checker.initiate_chat(
    math_asker,
    message="Is 97 a prime number? Use the tool before answering.",
    max_turns=3,
    summary_method="reflection_with_llm",
)

print(result.summary)


math_checker (to math_asker):

Is 97 a prime number? Use the tool before answering.

--------------------------------------------------------------------------------
math_asker (to math_checker):

***** Suggested tool call (call_HXPLxtPaOuN8llMOY3ZUgrT0): is_prime *****
Arguments: 
{"n":97}
*************************************************************************

--------------------------------------------------------------------------------

>>>>>>>> EXECUTING FUNCTION is_prime...
Call ID: call_HXPLxtPaOuN8llMOY3ZUgrT0
Input arguments: {'n': 97}

>>>>>>>> EXECUTED FUNCTION is_prime...
Call ID: call_HXPLxtPaOuN8llMOY3ZUgrT0
Input arguments: {'n': 97}
Output:
Yes
math_checker (to math_asker):

***** Response from calling tool (call_HXPLxtPaOuN8llMOY3ZUgrT0) *****
Yes
**********************************************************************

--------------------------------------------------------------------------------
math_asker (to math_checker):

Yes. 97 is a prime number.

Reason: I

### What is `register_function(...)` ?

In AG2, `register_function(...)` is used to expose a Python function as a **tool** that can be executed by one agent on behalf of another. This enables agents to delegate tasks like computation, data processing, or external API calls.

#### Purpose
- Extend agent capabilities beyond text generation.
- Allow agents to solve tasks through **function execution**.
- Enable collaborative workflows between a **caller** and an **executor** agent.

#### Parameters
- **function**: A regular Python function to be used as a tool.
- **caller**: The agent that will request the tool to be used.
- **executor**: The agent that will actually execute the function.
- **description** *(optional)*: A natural language description of the function for the LLM to decide when to use it.

#### Example
```python
register_function(
    is_prime,
    caller=math_asker,
    executor=math_checker,
    description="Check if a number is prime. Returns Yes or No."
)


## Structured Outputs<a name="structured-outputs"></a>

Structured outputs ensure consistent, validated agent responses using Pydantic models.   

This is crucial for:
  
- **Data validation**: Ensuring response format consistency
- **API integration**: Reliable data exchange
- **Quality assurance**: Preventing malformed outputs
- **Type safety**: Clear data contracts

**Analogy:** Like standardized hospital forms ensure doctors always record patient info the same way, structured outputs make sure agents respond in predictable, machine-readable formats.


In AG2, structured outputs are implemented using Pydantic models and the `response_format` parameter in the `LLMConfig`.

To ensure that your agent always returns outputs in a consistent structure, you define a Pydantic class (for example, `ResponseModel`) and assign it to the `response_format` argument of your configuration.

This tells the underlying LLM to return a JSON-compatible response matching the defined schema.

```python
class ResponseModel(BaseModel):
    name: str
    status: str

llm_config = LLMConfig(
    api_type="openai",
    model="gpt-4o-mini",
    response_format=ResponseModel
)
```

With this setup:

- The agent automatically formats its responses to match the ResponseModel. 
- You don’t need to prompt the LLM to format its response.
- The response is parsed and validated by AG2.

This approach is essential for reliable automation, integrations, and downstream processing.

Let's implement structured outputs with AG2:


In [13]:
# Define structured output model
class TicketSummary(BaseModel):
    customer_name: str
    issue_type: str
    urgency_level: str
    recommended_action: str

structured_llm_config = LLMConfig(
    {
        "api_type": "openai",
        "model": MODEL_NAME,
        "api_key": OPENAI_API_KEY,
    },
    response_format=TicketSummary,
)

support_agent = ConversableAgent(
    name="support_agent",
    system_message="Return only valid JSON that matches the requested support ticket schema.",
    llm_config=structured_llm_config,
    human_input_mode="NEVER",
)

support_user = ConversableAgent(name="support_user", llm_config=False, human_input_mode="NEVER")

result = support_user.initiate_chat(
    support_agent,
    message=(
        "Summarize this ticket: Jordan cannot log in after enabling MFA. "
        "They are blocked from payroll approval today."
    ),
    max_turns=2,
)

raw_content = result.chat_history[-1]["content"]
ticket = TicketSummary.model_validate_json(raw_content)
print(json.dumps(ticket.model_dump(), indent=2))


support_user (to support_agent):

Summarize this ticket: Jordan cannot log in after enabling MFA. They are blocked from payroll approval today.

--------------------------------------------------------------------------------
support_agent (to support_user):

{"customer_name":"Jordan","issue_type":"MFA / Authentication failure","urgency_level":"Urgent","recommended_action":"Unlock Jordan's account, reset or re-enroll MFA, provide temporary access to payroll approval if needed, verify MFA device/app is functioning, check for system outages, log incident for security review, and follow up with user to confirm login is restored."}

--------------------------------------------------------------------------------
support_user (to support_agent):



--------------------------------------------------------------------------------
support_agent (to support_user):

{"customer_name":"Jordan","issue_type":"MFA / Authentication failure","urgency_level":"Urgent","recommended_action":"Unlock Jordan'

## Best Practices<a name="best-practices"></a>

Based on our exploration of AG2 concepts, here are key best practices for building robust agent systems:

### Configuration and security
- **Never hardcode API keys** - use environment variables
- **Use config lists** for production systems with fallback models
- **Set appropriate temperature** values (0.0 for deterministic, 0.7-1.0 for creative)
- **Implement rate limiting** and error handling

### Agent design
- **Write clear system messages** that define role, capabilities, and constraints
- **Set max_consecutive_auto_reply** to prevent infinite loops
- **Choose appropriate human_input_mode** based on use case
- **Specialize agents** for specific tasks rather than creating generalists

### HITL implementation
- **Use HITL for high-stakes decisions** requiring human judgment
- **Implement clear escalation criteria** (amount thresholds, risk levels)
- **Provide context** to human supervisors for informed decisions
- **Log all human interventions** for audit trails

### Multi-agent orchestration
- **Design clear workflows** with defined handoffs between agents
- **Use GroupChat** for collaborative problem-solving
- **Implement termination conditions** to prevent endless conversations
- **Monitor conversation quality** and intervention points

### Tools and integration
- **Create focused tools** for specific capabilities
- **Implement proper error handling** in tool functions
- **Validate tool inputs** and outputs
- **Document tool capabilities** clearly for agents

### Structured outputs
- **Use Pydantic models** for data validation
- **Define clear schemas** for consistent outputs
- **Implement proper validation** with meaningful error messages
- **Version your schemas** for backward compatibility


## Conclusion<a name="conclusion"></a>

Congratulations! You've completed a comprehensive tour of AG2's basic concepts. Let's recap what we've learned:

### Key concepts mastered:

- **LLM Configuration** - The foundation that connects agents to language models
- **ConversableAgent** - The core building block for all AG2 agents
- **Human in the Loop (HITL)** - Enabling human oversight in AI workflows
- **Agent Orchestration** - Coordinating multiple agents for complex tasks
- **Tools & Extensions** - Extending agent capabilities beyond text
- **Structured Outputs** - Ensuring consistent, validated responses

### Practical skills developed:

- Creating and configuring specialized agents
- Implementing multi-agent collaboration patterns
- Building HITL workflows for critical decisions
- Integrating tools and external capabilities
- Validating outputs with structured schemas
- Following security and operational best practices

### Next steps:

Now that you understand AG2's basic concepts, you can:

1. **Build Domain-Specific Applications** - Apply these concepts to your specific use case.
2. **Explore Advanced Features** - Dive into code execution, web scraping, and API integrations.
3. **Scale Your Systems** - Learn about production deployment and monitoring.
4. **Join the Community** - Contribute to the AG2 ecosystem and learn from others.

### Things to remember:

- **Start simple** and gradually add complexity.
- **Test thoroughly** with various scenarios.
- **Monitor performance** and adjust configurations.
- **Follow best practices** for security and reliability.
- **Keep learning** as AG2 continues to evolve.

AG2 provides a powerful foundation for building intelligent, collaborative AI systems. The concepts you've learned here will serve as building blocks for more advanced implementations.  

### Advanced agentic design patterns:  
  
AG2 supports advanced features like custom rules for conversation endings, RAG, code execution, and secure tool use. You can learn more from the official documentation.  
  
Happy building with AG2! 🤖✨


## Authors


[Faranak Heidari](https://www.linkedin.com/in/faranakhdr/) is a Data Scientist and AI developr at IBM with expertise in GenAI, machine learning, and data analytics. Experienced in building LLMs, forecasting models, and scalable ML pipelines for domains like healthcare. Passionate about driving innovation and collaborating with teams to integrate AI into real-world workflows.

[Joshua Zhou](https://www.linkedin.com/in/joshuazhou1/) is a Data Scientist Intern at IBM.


## Change Log

<details>
    <summary>Click here for the changelog</summary>

|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
|-|-|-|-|
|2025-07-17|0.1|Faranak Heidari|Initial version created|
|2025-07-27|0.2|Steve Ryan|ID review and format fixes|
|2025-07-28|0.3|Mercedes Schneider|QA pass with edits|
</details>

---


Copyright © IBM Corporation. All rights reserved.
